# SoftMoE-Retrieval trên Kaggle
Bật **GPU** + **Internet ON**. Mỗi bộ một model riêng.

## 1. Add Data
Thêm: CUB-200-2011, Stanford Cars (có `cars_annos.mat`), DeepFashion In-Shop (có `list_eval_partition.txt`).

In [ ]:
!pip -q install transformers pytorch-metric-learning scipy timm 2>/dev/null
import torch; print('CUDA:',torch.cuda.is_available())

## 2. Lấy code
Upload thư mục `softmoe_retrival` thành Kaggle Dataset, hoặc git clone.

In [ ]:
import os,sys,glob,shutil
REPO='/kaggle/input/softmoe-retrival/softmoe_retrival'  # sửa slug nếu upload làm Kaggle Dataset
if not os.path.isdir(REPO):
    !git clone https://github.com/trong5nhan6/retrieval.git /kaggle/working/softmoe_retrival
    REPO='/kaggle/working/softmoe_retrival'
WORK='/kaggle/working/softmoe_retrival'
if REPO!=WORK: shutil.copytree(REPO,WORK,dirs_exist_ok=True)
sys.path.insert(0,WORK); os.chdir(WORK); print('repo at',WORK)

## 3. Tự dò path dataset

In [ ]:
from config import HCFG
def find(p):
    h=glob.glob(f'/kaggle/input/**/{p}',recursive=True); return os.path.dirname(h[0]) if h else None
HCFG.data_roots={k:v for k,v in {'cub':find('images.txt'),'cars':find('cars_annos.mat'),'inshop':find('list_eval_partition.txt')}.items() if v}
[print(k,'->',v) for k,v in HCFG.data_roots.items()]

## 4. Smoke-test

In [ ]:
from models.hybrid_encoder import HybridEncoder
from models.hyms_route import HyMSRoute
dev='cuda' if torch.cuda.is_available() else 'cpu'
enc=HybridEncoder(HCFG.vit_name,HCFG.cnn_name,dev); enc.freeze_all()
m=HyMSRoute(enc,HCFG).to(dev); z,rho,C=m(torch.randn(2,3,224,224).to(dev))
print('z',z.shape,'rho',rho.shape,'C',C.shape); del m,enc; torch.cuda.empty_cache()

## 5. Train

In [ ]:
DATASET='cub'  # 'cub'|'cars'|'inshop'
for s in [42]:
    !python train.py --dataset {DATASET} --seed {s} --epochs 60

In [ ]:
!ls -la results/checkpoints results/ 2>/dev/null